In [0]:
%run "../../utils"

In [0]:
# Load raw bronze table
df_bronze = spark.table(f"{bronze_schema}.alt_fuel_stations_raw")

In [0]:
# Transform and clean data for silver table
df_silver = (
    df_bronze
      .withColumn("station_id", expr("try_cast(id as BIGINT)"))  # Cast id to BIGINT
      .filter(col("station_id").isNotNull())  # Keep only valid station_id
      .select(
          col("station_id"),
          col("station_name"),
          col("city"),
          col("state"),
          col("fuel_type_code"),
          expr("try_cast(latitude as DOUBLE)").alias("latitude"),
          expr("try_cast(longitude as DOUBLE)").alias("longitude"),
          to_date(col("open_date")).alias("open_date"),
          col("access_code"),
          col("facility_type"),
          col("owner_type_code"),
          current_timestamp().alias("ingestion_ts")
      )
      .dropDuplicates(["station_id"])  # Remove duplicate station_id
)

# Write cleaned data to silver Delta table
df_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{silver_schema}.alt_fuel_stations")


In [0]:
# Extract records with invalid station_id for quarantine
df_bad = (
    df_bronze
      .withColumn("station_id", expr("try_cast(id as BIGINT)"))  # Attempt to cast id to BIGINT
      .filter(col("station_id").isNull())  # Filter rows where station_id is invalid
      .withColumn("dq_reason", expr("concat('invalid id: ', id)"))  # Add data quality reason
      .withColumn("ingestion_ts", current_timestamp())  # Add ingestion timestamp
)

# Write quarantined records to Delta table
df_bad.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{silver_schema}.alt_fuel_stations_quarantine")